# KNN

1) Import Necessary Package

In [18]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA


OUTPUT_DIR = "../outputs"
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
TAB_DIR = os.path.join(OUTPUT_DIR, "tables")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

2. Data Preparation for Modelling

In [24]:
# read the processed dataset
df = pd.read_csv(r"../data/processed/feature_engineered_with_SLM.csv")

In [25]:
# print the shape of the dataset
df.shape

(1200, 33)

In [26]:
# no columns are hidden
pd.set_option('display.max_columns', None)
# dislay first 5 rows of the dataset
df.head()

,transaction_id,customer_id,transaction_datetime,country,channel,merchant_category,amount,device_trust_score,num_txn_24h_customer,previous_chargeback_count,is_fraud,transaction_note,slm_risk_score,slm_urgency,slm_anomaly_category,txn_hour,txn_dayofweek,txn_week,is_weekend,is_night,prev_txn_time,txn_gap_minutes,txn_7d_count,txn_hour_count,txn_dayofweek_count,customer_mean_amount,customer_std_amount,amount_z_customer,is_risky_country,low_device_trust,high_amount_z,composite_risk_score,urgency_numeric
0,998,1000,2022-10-05 15:13:00,US,mobile_app,luxury,672.47,23.0,2,3,0,new device,9,high,fraud,15,2,40,0,0,NaN,-1.000000,NaN,1,2,1931.283333,1662.07937,-0.757373,0,1,0,7.333333,2
1,953,1000,2023-02-05 06:18:18,US,web,gaming,2593.31,84.0,0,0,1,overnight shipping,8,high,fraud,6,6,5,1,0,2022-10-05 15:13:00,176585.300000,NaN,1,2,1931.283333,1662.07937,0.398312,0,0,0,3.555556,2
2,94,1000,2023-03-23 18:04:02,US,pos_terminal,electronics,302.22,70.0,3,1,0,high value item,9,high,fraud,18,3,12,0,0,2023-02-05 06:18:18,66945.733333,NaN,1,2,1931.283333,1662.07937,-0.980136,0,0,0,4.000000,2
3,457,1000,2023-04-05 00:14:27,US,mobile_app,gaming,4395.07,24.0,0,2,0,repeat purchase,9,high,fraud,0,2,14,0,1,2023-03-23 18:04:02,17650.416667,NaN,1,2,1931.283333,1662.07937,1.482352,0,1,0,7.333333,2
4,1199,1000,2023-07-13 02:02:46,US,mobile_app,luxury,579.99,23.0,1,0,0,overnight shipping,5,low,unknown,2,3,28,0,1,2023-04-05 00:14:27,142668.316667,NaN,1,2,1931.283333,1662.07937,-0.813014,0,1,0,5.555556,0


In [27]:
# check the distribution of the target variable
df['is_fraud'].value_counts()

is_fraud
0    1119
1      81
Name: count, dtype: int64

In [28]:
# check positive rate
print("Positive rate:", df['is_fraud'].mean())

Positive rate: 0.0675


In [29]:
# extra cleaning will be done later

3. Split Data

In [30]:
# define variables and drop unnecessary columns
X = df.drop(columns=['is_fraud', 'transaction_id', 'customer_id', 'transaction_datetime','prev_txn_time','txn_7d_count','transaction_note'])
y = df['is_fraud']


In [31]:
# split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


4. Preprocess

In [ ]:
# identify categorical and numerical columns
categorical_cols = [c for c in X.columns if X[c].dtype == "object"]
numeric_cols = [c for c in X.columns if c not in categorical_cols]

In [33]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    # OPTIONAL: comment out PCA if you want no dimensionality reduction
    ("pca", PCA(n_components=0.95, random_state=42))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ], remainder="drop")



In [34]:
X_train = preprocess.fit_transform(X_train)
X_test = preprocess.transform(X_test)

print(X_train.shape)
print(X_test.shape)


(960, 37)
(240, 37)


5. Modelling